# Future slowdown-label creation

This notebook recreates provisional Rule C as `slowdown_now`, then asks whether a later slowdown occurs within valid 5- and 10-minute horizons. It does not build features, split data, or train a model.

### 1. Define paths and fingerprint protected datasets

**What the cell does:** Imports libraries, defines repository-relative paths, and calculates SHA-256 checksums for the segmented input and protected cleaned dataset.  
**Why it is important:** Label creation must be reproducible and must never overwrite source datasets.  
**What to understand:** The printed fingerprints identify the exact inputs and will be checked again after all outputs are saved.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SEGMENTED_INPUT_PATH = PROJECT_ROOT / "data" / "interim" / "segmented_metrics.csv"
CLEANED_PROTECTED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_metrics.csv"
LABELED_OUTPUT_PATH = PROJECT_ROOT / "data" / "interim" / "labeled_metrics.csv"
SUMMARY_OUTPUT_PATH = PROJECT_ROOT / "reports" / "future_label_summary.csv"
MACHINE_OUTPUT_PATH = PROJECT_ROOT / "reports" / "future_label_by_machine.csv"
RUN_OUTPUT_PATH = PROJECT_ROOT / "reports" / "future_label_by_run.csv"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

protected_paths = [SEGMENTED_INPUT_PATH, CLEANED_PROTECTED_PATH]
missing_paths = [str(path) for path in protected_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Required datasets are missing: {missing_paths}")

hashes_before = {path: sha256_file(path) for path in protected_paths}
for path, fingerprint in hashes_before.items():
    print(f"{path}: {fingerprint}")

/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/interim/segmented_metrics.csv: eafbd4f01ddf3f1b7bd5d0df11ce4b4e467c679e7718539ddbdf39b4ce6a8578
/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/processed/cleaned_metrics.csv: 801976ddbe91a7415ce84038ff7df5b89c601e2201e56eaf5503154f93c2e577


### 2. Load and verify the segmented dataset

**What the cell does:** Loads a copy, verifies identity/resource columns, parses UTC timestamps safely, and stably sorts by machine, run, segment, and time.  
**Why it is important:** Both rolling pressure calculations and future searches require valid chronological sequences with explicit boundaries.  
**What to understand:** Every row has a valid machine/run/segment/timestamp identity, and only the analysis copy is sorted.

In [2]:
segmented_input = pd.read_csv(SEGMENTED_INPUT_PATH)
required_identity = {"machine_id", "run_id", "segment_id", "timestamp"}
resource_columns = {"cpu_pct", "ram_pct", "swap_pct", "disk_latency_ms", "context_switches_per_s"}
missing_columns = (required_identity | resource_columns) - set(segmented_input.columns)
if missing_columns:
    raise KeyError(f"Required columns are missing: {sorted(missing_columns)}")

original_columns = segmented_input.columns.tolist()
label_data = segmented_input.copy(deep=True)
label_data["_source_row"] = np.arange(len(label_data))
label_data["_timestamp_dt"] = pd.to_datetime(label_data["timestamp"], errors="coerce", utc=True)
if label_data["_timestamp_dt"].isna().any():
    raise ValueError(f"Invalid timestamps found: {label_data['_timestamp_dt'].isna().sum()}")
if label_data[list(required_identity)].isna().any().any():
    raise ValueError("Machine, run, segment, or timestamp identity contains missing values.")

sequence_keys = ["machine_id", "run_id", "segment_id"]
label_data = label_data.sort_values(
    sequence_keys + ["_timestamp_dt", "_source_row"], kind="mergesort"
).copy()

print(f"Rows loaded: {len(label_data):,}")
print(f"Machines: {label_data['machine_id'].nunique()}")
print(f"Runs: {label_data['run_id'].nunique()}")
print(f"Segments: {label_data['segment_id'].nunique()}")

Rows loaded: 31,059
Machines: 3
Runs: 10
Segments: 367


### 3. Reproduce the exact provisional Rule C configuration

**What the cell does:** Centralizes the same moderate and severe thresholds used in `06_label_definition.ipynb`.  
**Why it is important:** Future targets must be based on the identical `slowdown_now` definition; changing a threshold would invalidate comparisons.  
**What to understand:** Moderate signals use trailing 30-second means, severe signals use trailing 60-second means, and both require at least three observations.

In [3]:
RULE_C_CONFIG = {
    "moderate_window_seconds": 30,
    "severe_window_seconds": 60,
    "minimum_observations": 3,
    "moderate": {
        "cpu_pct_mean": 80.0,
        "ram_pct_mean": 85.0,
        "swap_pct_mean": 60.0,
        "disk_latency_ms_mean": 5.0,
        "context_run_quantile": 0.95,
        "context_median_multiplier": 1.5,
    },
    "severe": {
        "cpu_pct_mean": 95.0,
        "ram_pct_mean": 95.0,
        "swap_pct_mean": 70.0,
        "disk_latency_ms_mean": 20.0,
        "context_run_quantile": 0.99,
        "context_median_multiplier": 2.0,
    },
}

display(pd.json_normalize(RULE_C_CONFIG, sep=".").T.rename(columns={0: "value"}))
print("Rule C: slowdown_now = one severe signal OR at least two moderate signals.")

,value
moderate_window_seconds,30.00
severe_window_seconds,60.00
minimum_observations,3.00
moderate.cpu_pct_mean,80.00
moderate.ram_pct_mean,85.00
moderate.swap_pct_mean,60.00
moderate.disk_latency_ms_mean,5.00
moderate.context_run_quantile,0.95
moderate.context_median_multiplier,1.50
severe.cpu_pct_mean,95.00


Rule C: slowdown_now = one severe signal OR at least two moderate signals.


### 4. Recreate bounded 30- and 60-second rolling means

**What the cell does:** Calculates trailing resource means independently inside each machine/run/segment and creates run-relative context-switch baselines.  
**Why it is important:** No pressure calculation may cross a collection gap, run, or machine boundary.  
**What to understand:** The rolling logic includes only the current and earlier rows in the same segment and matches the previous notebook.

In [4]:
rolling_metrics = ["cpu_pct", "ram_pct", "swap_pct", "disk_latency_ms", "context_switches_per_s"]
for window_name, seconds in [
    ("moderate", RULE_C_CONFIG["moderate_window_seconds"]),
    ("severe", RULE_C_CONFIG["severe_window_seconds"]),
]:
    for metric in rolling_metrics:
        label_data[f"{metric}_{window_name}_mean"] = np.nan
    for _, segment in label_data.groupby(sequence_keys, sort=False):
        ordered = segment.sort_values("_timestamp_dt")
        indexed = ordered.set_index("_timestamp_dt")
        for metric in rolling_metrics:
            rolled = indexed[metric].rolling(
                f"{seconds}s", min_periods=RULE_C_CONFIG["minimum_observations"]
            ).mean()
            label_data.loc[ordered.index, f"{metric}_{window_name}_mean"] = rolled.to_numpy()

context_baselines = (
    label_data.groupby(["machine_id", "run_id"])["context_switches_per_s"]
    .agg(
        context_run_median="median",
        context_run_p95=lambda values: values.quantile(RULE_C_CONFIG["moderate"]["context_run_quantile"]),
        context_run_p99=lambda values: values.quantile(RULE_C_CONFIG["severe"]["context_run_quantile"]),
    )
    .reset_index()
)
label_data = label_data.merge(context_baselines, on=["machine_id", "run_id"], how="left", validate="many_to_one")

display(label_data[sequence_keys + ["timestamp", "cpu_pct_moderate_mean", "cpu_pct_severe_mean"]].head())

,machine_id,run_id,segment_id,timestamp,cpu_pct_moderate_mean,cpu_pct_severe_mean
0,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202__6835f125-a038-4092-b...,2026-07-25T14:45:59.803000Z,NaN,NaN
1,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202__6835f125-a038-4092-b...,2026-07-25T14:46:00.857000Z,NaN,NaN
2,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202__6835f125-a038-4092-b...,2026-07-25T14:46:02.829000Z,5.066667,5.066667
3,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202__6835f125-a038-4092-b...,2026-07-25T14:46:04.845000Z,4.950000,4.950000
4,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202__6835f125-a038-4092-b...,2026-07-25T14:46:06.853000Z,5.420000,5.420000


### 5. Recreate `slowdown_now` using Rule C

**What the cell does:** Builds five moderate and five severe sustained-resource signals, then sets `slowdown_now=1` for one severe signal or at least two moderate signals.  
**Why it is important:** The future target must predict the same provisional present-time event definition chosen for manual review.  
**What to understand:** Only CPU, RAM, swap, disk latency, and context switching contribute; timing, timeout, status, temperature, and GPU fields do not.

In [5]:
moderate = RULE_C_CONFIG["moderate"]
severe = RULE_C_CONFIG["severe"]

label_data["moderate_cpu"] = label_data["cpu_pct_moderate_mean"].ge(moderate["cpu_pct_mean"])
label_data["moderate_ram"] = label_data["ram_pct_moderate_mean"].ge(moderate["ram_pct_mean"])
label_data["moderate_swap"] = label_data["swap_pct_moderate_mean"].ge(moderate["swap_pct_mean"])
label_data["moderate_disk"] = label_data["disk_latency_ms_moderate_mean"].ge(moderate["disk_latency_ms_mean"])
label_data["moderate_context"] = (
    label_data["context_switches_per_s_moderate_mean"].ge(label_data["context_run_p95"])
    & label_data["context_switches_per_s_moderate_mean"].ge(
        moderate["context_median_multiplier"] * label_data["context_run_median"]
    )
)
label_data["severe_cpu"] = label_data["cpu_pct_severe_mean"].ge(severe["cpu_pct_mean"])
label_data["severe_ram"] = label_data["ram_pct_severe_mean"].ge(severe["ram_pct_mean"])
label_data["severe_swap"] = label_data["swap_pct_severe_mean"].ge(severe["swap_pct_mean"])
label_data["severe_disk"] = label_data["disk_latency_ms_severe_mean"].ge(severe["disk_latency_ms_mean"])
label_data["severe_context"] = (
    label_data["context_switches_per_s_severe_mean"].ge(label_data["context_run_p99"])
    & label_data["context_switches_per_s_severe_mean"].ge(
        severe["context_median_multiplier"] * label_data["context_run_median"]
    )
)

moderate_columns = ["moderate_cpu", "moderate_ram", "moderate_swap", "moderate_disk", "moderate_context"]
severe_columns = ["severe_cpu", "severe_ram", "severe_swap", "severe_disk", "severe_context"]
label_data["moderate_signal_count"] = label_data[moderate_columns].sum(axis=1).astype(int)
label_data["severe_signal_count"] = label_data[severe_columns].sum(axis=1).astype(int)
label_data["slowdown_now"] = (
    label_data["severe_signal_count"].ge(1)
    | label_data["moderate_signal_count"].ge(2)
).astype("int8")

prohibited_inputs = {
    "missed_deadline", "sensor_errors_json", "status", "ended_at_utc", "run_complete",
    "temperature_c", "gpu_usage_pct", "diagnostic_class", "revised_diagnostic_class"
}
assert not (set(rolling_metrics) & prohibited_inputs)
print(f"slowdown_now positive rows: {label_data['slowdown_now'].sum():,} ({label_data['slowdown_now'].mean() * 100:.4f}%)")

slowdown_now positive rows: 1,931 (6.2172%)


### 6. Create strict future-only 5- and 10-minute labels

**What the cell does:** Searches each segment strictly after the current timestamp and through the horizon endpoint. It assigns missing when the segment does not extend through the complete horizon.  
**Why it is important:** Using the current event would leak the answer, crossing a boundary would mix unrelated time, and replacing incomplete horizons with zero would create false negatives.  
**What to understand:** A valid label is 1 if any later `slowdown_now=1` occurs within the complete horizon, otherwise 0; invalid segment-tail rows remain `NaN`.

In [6]:
def add_future_label(dataframe, horizon_minutes):
    horizon = pd.Timedelta(minutes=horizon_minutes)
    validity_column = f"valid_{horizon_minutes}min_horizon"
    label_column = f"slowdown_in_{horizon_minutes}min"
    dataframe[validity_column] = False
    dataframe[label_column] = pd.Series(pd.NA, index=dataframe.index, dtype="Float64")

    for _, segment in dataframe.groupby(sequence_keys, sort=False):
        ordered = segment.sort_values("_timestamp_dt")
        indices = ordered.index.to_numpy()
        times = (
            ordered["_timestamp_dt"].dt.tz_convert("UTC").dt.tz_localize(None)
            .to_numpy(dtype="datetime64[ns]")
        )
        events = ordered["slowdown_now"].to_numpy(dtype=np.int8)
        cumulative_events = np.concatenate(([0], np.cumsum(events)))
        horizon_delta = np.timedelta64(horizon_minutes, "m")
        segment_end = times[-1]

        valid = segment_end >= times + horizon_delta
        left_positions = np.searchsorted(times, times, side="right")
        right_positions = np.searchsorted(times, times + horizon_delta, side="right")
        future_event_counts = cumulative_events[right_positions] - cumulative_events[left_positions]
        labels = (future_event_counts > 0).astype(float)

        dataframe.loc[indices, validity_column] = valid
        valid_indices = indices[valid]
        dataframe.loc[valid_indices, label_column] = labels[valid]

    dataframe[validity_column] = dataframe[validity_column].astype(bool)
    return validity_column, label_column

valid_5_column, label_5_column = add_future_label(label_data, 5)
valid_10_column, label_10_column = add_future_label(label_data, 10)

assert label_data.loc[~label_data[valid_5_column], label_5_column].isna().all()
assert label_data.loc[~label_data[valid_10_column], label_10_column].isna().all()
print(f"Valid 5-minute horizons: {label_data[valid_5_column].sum():,}")
print(f"Valid 10-minute horizons: {label_data[valid_10_column].sum():,}")

Valid 5-minute horizons: 26,036
Valid 10-minute horizons: 22,980


### 7. Validate future-label boundaries and strictness

**What the cell does:** Rechecks that labels are missing at incomplete segment tails and spot-validates positive labels against strictly later events inside the same sequence.  
**Why it is important:** Explicit assertions catch accidental current-row inclusion, boundary crossing, or invalid-horizon zeros.  
**What to understand:** Passing checks mean the generated targets obey the causal and segmentation rules.

In [7]:
for horizon_minutes, validity_column, label_column in [
    (5, valid_5_column, label_5_column),
    (10, valid_10_column, label_10_column),
]:
    assert label_data.loc[label_data[validity_column], label_column].isin([0.0, 1.0]).all()
    assert label_data.loc[~label_data[validity_column], label_column].isna().all()
    positive_examples = label_data.loc[label_data[label_column].eq(1.0)].head(100)
    for _, row in positive_examples.iterrows():
        current_time = row["_timestamp_dt"]
        horizon_end = current_time + pd.Timedelta(minutes=horizon_minutes)
        future_events = label_data.loc[
            label_data["machine_id"].eq(row["machine_id"])
            & label_data["run_id"].eq(row["run_id"])
            & label_data["segment_id"].eq(row["segment_id"])
            & label_data["_timestamp_dt"].gt(current_time)
            & label_data["_timestamp_dt"].le(horizon_end)
            & label_data["slowdown_now"].eq(1)
        ]
        assert len(future_events) > 0

print("Boundary, strict-future, and invalid-horizon checks passed.")

Boundary, strict-future, and invalid-horizon checks passed.


### 8. Summarize labels overall, by machine, and by run

**What the cell does:** Counts valid/invalid rows, positives, negatives, rates, and positive coverage for both horizons at overall, machine, and run levels.  
**Why it is important:** A target must have sufficient valid examples and should not be confined to one machine or run.  
**What to understand:** The 5-minute results are the main target evaluation; 10-minute results are comparison only.

In [8]:
def horizon_counts(dataframe, horizon_minutes):
    validity = dataframe[f"valid_{horizon_minutes}min_horizon"]
    labels = dataframe[f"slowdown_in_{horizon_minutes}min"]
    valid_rows = int(validity.sum())
    positive_rows = int(labels.eq(1).sum())
    negative_rows = int(labels.eq(0).sum())
    return {
        f"valid_{horizon_minutes}min_rows": valid_rows,
        f"invalid_{horizon_minutes}min_rows": int(len(dataframe) - valid_rows),
        f"positive_{horizon_minutes}min_rows": positive_rows,
        f"negative_{horizon_minutes}min_rows": negative_rows,
        f"positive_{horizon_minutes}min_percentage": round(positive_rows / valid_rows * 100, 4) if valid_rows else np.nan,
    }

summary_records = []
for horizon_minutes in [5, 10]:
    counts = horizon_counts(label_data, horizon_minutes)
    label_column = f"slowdown_in_{horizon_minutes}min"
    positive_data = label_data.loc[label_data[label_column].eq(1)]
    summary_records.append({
        "horizon_minutes": horizon_minutes,
        "target_role": "main_target" if horizon_minutes == 5 else "comparison_only",
        "total_rows": int(len(label_data)),
        "valid_rows": counts[f"valid_{horizon_minutes}min_rows"],
        "invalid_rows_near_segment_ends": counts[f"invalid_{horizon_minutes}min_rows"],
        "positive_rows": counts[f"positive_{horizon_minutes}min_rows"],
        "negative_rows": counts[f"negative_{horizon_minutes}min_rows"],
        "positive_percentage": counts[f"positive_{horizon_minutes}min_percentage"],
        "machines_with_positive_labels": int(positive_data["machine_id"].nunique()),
        "total_machines": int(label_data["machine_id"].nunique()),
        "runs_with_positive_labels": int(positive_data["run_id"].nunique()),
        "total_runs": int(label_data["run_id"].nunique()),
    })
future_label_summary = pd.DataFrame(summary_records)

def grouped_distribution(dataframe, keys):
    rows = []
    for group_key, group in dataframe.groupby(keys, dropna=False):
        key_values = group_key if isinstance(group_key, tuple) else (group_key,)
        record = dict(zip(keys, key_values))
        record["total_rows"] = int(len(group))
        record["slowdown_now_positive_rows"] = int(group["slowdown_now"].sum())
        record.update(horizon_counts(group, 5))
        record.update(horizon_counts(group, 10))
        rows.append(record)
    return pd.DataFrame(rows)

future_label_by_machine = grouped_distribution(label_data, ["machine_id"])
future_label_by_run = grouped_distribution(label_data, ["machine_id", "run_id"])

display(future_label_summary)
print("By machine:")
display(future_label_by_machine)
print("By run:")
display(future_label_by_run)

,horizon_minutes,target_role,total_rows,valid_rows,invalid_rows_near_segment_ends,positive_rows,negative_rows,positive_percentage,machines_with_positive_labels,total_machines,runs_with_positive_labels,total_runs
0,5,main_target,31059,26036,5023,3643,22393,13.9922,3,3,8,10
1,10,comparison_only,31059,22980,8079,4259,18721,18.5335,3,3,8,10


By machine:


,machine_id,total_rows,slowdown_now_positive_rows,valid_5min_rows,invalid_5min_rows,positive_5min_rows,negative_5min_rows,positive_5min_percentage,valid_10min_rows,invalid_10min_rows,positive_10min_rows,negative_10min_rows,positive_10min_percentage
0,0890dcc046c079acc4de4202,12386,114,11933,453,600,11333,5.0281,11484,902,1051,10433,9.1519
1,7232bc533c21ce408d45d473,7979,966,4197,3782,1188,3009,28.3059,2178,5801,873,1305,40.0826
2,a0f8c86097e55fbfa506d057,10694,851,9906,788,1855,8051,18.7260,9318,1376,2335,6983,25.0590


By run:


,machine_id,run_id,total_rows,slowdown_now_positive_rows,valid_5min_rows,invalid_5min_rows,positive_5min_rows,negative_5min_rows,positive_5min_percentage,valid_10min_rows,invalid_10min_rows,positive_10min_rows,negative_10min_rows,positive_10min_percentage
0,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,3600,9,3449,151,344,3105,9.9739,3299,301,644,2655,19.5211
1,0890dcc046c079acc4de4202,89cdc34b-e02b-43b4-9284-144276df508a,8786,105,8484,302,256,8228,3.0174,8185,601,407,7778,4.9725
2,7232bc533c21ce408d45d473,19127a70-e60c-4b47-b3e6-71e89175c174,211,45,119,92,105,14,88.2353,3,208,3,0,100.0000
3,7232bc533c21ce408d45d473,d88b15dd-1915-43ca-90ef-69f31ff4d9c1,2548,303,1165,1383,311,854,26.6953,329,2219,117,212,35.5623
4,7232bc533c21ce408d45d473,ec61755d-b5be-42ac-875d-3123e92add7a,2642,115,1302,1340,62,1240,4.7619,694,1948,17,677,2.4496
5,7232bc533c21ce408d45d473,fac82c2e-545a-402a-80f8-3d1fccb72c68,2578,503,1611,967,710,901,44.0720,1152,1426,736,416,63.8889
6,a0f8c86097e55fbfa506d057,2e9f4457-2a7f-4647-87ed-b86ac6343d33,377,0,182,195,0,182,0.0000,107,270,0,107,0.0000
7,a0f8c86097e55fbfa506d057,373070d1-ba59-4244-88ab-2d44a21f4983,422,2,261,161,0,261,0.0000,111,311,0,111,0.0000
8,a0f8c86097e55fbfa506d057,70577f8e-1430-4489-8602-2096521ab84e,5895,743,5603,292,1402,4201,25.0223,5387,508,1586,3801,29.4412
9,a0f8c86097e55fbfa506d057,f24e9c1a-f11e-405c-b663-45fa6e25c405,4000,106,3860,140,453,3407,11.7358,3713,287,749,2964,20.1724


### 9. Save the labeled dataset and summary reports

**What the cell does:** Saves every original segmented column plus `slowdown_now`, both validity flags, and both future labels, then reads all outputs back for validation.  
**Why it is important:** The dataset must preserve all rows and represent unavailable horizons as missing rather than false negatives.  
**What to understand:** Output row count equals the input, 5 minutes is identified as the main target, and 10 minutes remains comparison-only.

In [9]:
label_columns = [
    "slowdown_now", "valid_5min_horizon", "slowdown_in_5min",
    "valid_10min_horizon", "slowdown_in_10min"
]
labeled_output = label_data[original_columns + label_columns].copy()

LABELED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
RULE_OUTPUT_PATHS = [SUMMARY_OUTPUT_PATH, MACHINE_OUTPUT_PATH, RUN_OUTPUT_PATH]
for path in RULE_OUTPUT_PATHS:
    path.parent.mkdir(parents=True, exist_ok=True)

labeled_output.to_csv(LABELED_OUTPUT_PATH, index=False)
future_label_summary.to_csv(SUMMARY_OUTPUT_PATH, index=False)
future_label_by_machine.to_csv(MACHINE_OUTPUT_PATH, index=False)
future_label_by_run.to_csv(RUN_OUTPUT_PATH, index=False)

saved_labeled = pd.read_csv(LABELED_OUTPUT_PATH)
saved_summary = pd.read_csv(SUMMARY_OUTPUT_PATH)
saved_machine = pd.read_csv(MACHINE_OUTPUT_PATH)
saved_run = pd.read_csv(RUN_OUTPUT_PATH)

assert len(saved_labeled) == len(segmented_input)
assert saved_labeled.columns.tolist() == original_columns + label_columns
assert saved_labeled.loc[~saved_labeled["valid_5min_horizon"], "slowdown_in_5min"].isna().all()
assert saved_labeled.loc[~saved_labeled["valid_10min_horizon"], "slowdown_in_10min"].isna().all()
assert len(saved_summary) == 2

print(f"Created {LABELED_OUTPUT_PATH} with {len(saved_labeled):,} rows.")
print(f"Created {SUMMARY_OUTPUT_PATH}")
print(f"Created {MACHINE_OUTPUT_PATH}")
print(f"Created {RUN_OUTPUT_PATH}")

Created /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/interim/labeled_metrics.csv with 31,059 rows.
Created /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/future_label_summary.csv
Created /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/future_label_by_machine.csv
Created /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/future_label_by_run.csv


### 10. Assess provisional usability and verify non-modification

**What the cell does:** Applies transparent minimum usability checks to the 5-minute summary and recomputes protected-dataset checksums.  
**Why it is important:** The next ML step needs enough valid positive/negative rows across multiple machines and runs, while input integrity must be proven.  
**What to understand:** `usable_for_next_ml_step` is a data-coverage assessment, not approval of the provisional Rule C semantics.

In [10]:
main_summary = future_label_summary.loc[future_label_summary["horizon_minutes"].eq(5)].iloc[0]
usable_for_next_ml_step = bool(
    main_summary["valid_rows"] >= 1000
    and main_summary["positive_rows"] >= 100
    and main_summary["negative_rows"] >= 100
    and 1.0 <= main_summary["positive_percentage"] <= 30.0
    and main_summary["machines_with_positive_labels"] >= 2
    and main_summary["runs_with_positive_labels"] >= 2
)

hashes_after = {path: sha256_file(path) for path in protected_paths}
protected_inputs_unchanged = hashes_before == hashes_after

print("FINAL FUTURE-LABEL SUMMARY")
display(future_label_summary)
print(f"5-minute target usable for the next ML preparation step: {usable_for_next_ml_step}")
print("This checks coverage and class balance only; Rule C still requires semantic manual validation.")
print("No train/validation/test split, model features, normalization, imputation, or training was performed.")
print(f"Protected input datasets remained unchanged: {protected_inputs_unchanged}")

assert protected_inputs_unchanged, "A protected input dataset changed during future-label creation."


FINAL FUTURE-LABEL SUMMARY


,horizon_minutes,target_role,total_rows,valid_rows,invalid_rows_near_segment_ends,positive_rows,negative_rows,positive_percentage,machines_with_positive_labels,total_machines,runs_with_positive_labels,total_runs
0,5,main_target,31059,26036,5023,3643,22393,13.9922,3,3,8,10
1,10,comparison_only,31059,22980,8079,4259,18721,18.5335,3,3,8,10


5-minute target usable for the next ML preparation step: True
This checks coverage and class balance only; Rule C still requires semantic manual validation.
No train/validation/test split, model features, normalization, imputation, or training was performed.
Protected input datasets remained unchanged: True
